# 第 7 课 · BeautifulSoup 查找

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 上一课我们看懂了「HTML 是一棵树」。这一课请出第一位管理员 **BeautifulSoup（bs4）**，
> 它把字符串整理成树，再让你用一句话就找到节点。本课只引入：`soup`、`find / find_all`、按 class 找、Tag 三件套。

In [1]:
# 每本 notebook 开头都要重新 import（哪怕上一课写过）
from pathlib import Path
from bs4 import BeautifulSoup       # 第三方库，已在 ai_learn 里装好
import bs4
print('bs4 版本：', bs4.__version__)

DATA_DIR = Path(r'C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫\data')
html_text = (DATA_DIR / 'books_index.html').read_text(encoding='utf-8')

bs4 版本： 4.15.0


## 7.1 第一步：BeautifulSoup(html, 'lxml') 得到 soup

**生活化类比**：你抱着一摞散页（HTML 字符串）找到图书管理员。他不跟你闲聊，
先把散页按「谁套着谁」装订成一架可以随时抽取的书——这本书对象就叫 **soup**。

**是什么**：`BeautifulSoup(html_text, 'lxml')` 返回一个 soup 对象；`'lxml'` 是干活的解析器（负责读字符串）。

**为什么需要**：字符串只能 `find` 位置、切片，稍改页面就切错；soup 能按「标签名 / class」直接问，稳得多。

In [2]:
soup = BeautifulSoup(html_text, 'lxml')   # 把字符串交给管理员，整理成树
print('soup 的类型：', type(soup))
print('soup.title：', soup.title)         # 直接拿到 <title> 这个节点（对象）
print('soup.title.text：', soup.title.text)   # .text 是它里面的纯文字

soup 的类型： <class 'bs4.BeautifulSoup'>
soup.title： <title>在线书店 · 全部书籍（第 1 页）</title>
soup.title.text： 在线书店 · 全部书籍（第 1 页）


## 7.2 find 与 find_all：按标签名找

**生活化类比**：
- 「给我**第一个** h3」→ `find('h3')`；
- 「把**全部** article 都列出来」→ `find_all('article')`，返回一个列表。

**是什么**：`find(标签名)` 只返回第一个匹配节点；`find_all(标签名)` 返回所有匹配节点组成的列表。

**为什么需要**：网页里同种标签很多，你得能区分「只要一个」还是「全都要」。

In [3]:
first_h3 = soup.find('h3')              # 只返回第一个 <h3> 节点
print('第一个 h3：', first_h3)

all_articles = soup.find_all('article')  # 返回列表，里面是全部 4 个 <article>
print('article 个数：', len(all_articles))
print('列表里第二个节点的类型：', type(all_articles[1]))   # 也是一个 Tag 对象

第一个 h3： <h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the Attic</a></h3>
article 个数： 4
列表里第二个节点的类型： <class 'bs4.element.Tag'>


## 7.3 按 class 找，以及读属性

**生活化类比**：光按书名找不够，还得按「分类贴纸」找。网页上的 `class="product_pod"` 就是分类贴纸。

**是什么**：
- `find('article', class_='product_pod')` 按 class 找。注意写成 `class_`——因为 `class` 是 Python 保留字；
- 找到节点后，`节点.get('href')` 读它的属性（`节点['href']` 也可以）。

**为什么需要**：同名标签几十个，真正能唯一定位的往往是 class。

In [4]:
pod = soup.find('article', class_='product_pod')  # 找到第一个“书盒”
print('这个盒子的标签名 .name：', pod.name)

link = pod.find('a')                  # 在这个盒子内部再找 <a>
print("a 的 href 属性：", link.get('href'))
print("a 的 title 属性：", link.get('title'))

这个盒子的标签名 .name： article
a 的 href 属性： catalogue/a-light-in-the-attic_1000/index.html
a 的 title 属性： A Light in the Attic


## 7.4 Tag 对象三件套：.name / .get_text() / .get()

**是什么**：每个找到的节点（Tag）上最常用的三个动作：
- `.name`：它是什么标签（如 `'p'`）；
- `.get_text()`：它里面所有纯文字（自动去掉标签）；
- `.get('属性名')`：读某个属性；`.attrs` 能一次性看到全部属性。

**为什么需要**：抠数据最后都是落在「文字」和「属性」上，这三件套就是出口。

In [5]:
price_p = soup.find('p', class_='price_color')
print('标签名 .name      ：', price_p.name)
print('纯文字 .get_text()：', price_p.get_text())
print('全部属性 .attrs   ：', price_p.attrs)

标签名 .name      ： p
纯文字 .get_text()： ¥51.77
全部属性 .attrs   ： {'class': ['price_color']}


## 7.5 综合：遍历 4 个书盒，把书名 / 价格 / 星级一次抠出来

把前面几步拼起来：先 `find_all` 所有书盒，再在每个盒子里分别取书名、价格、星级。

- 书名：`h3 > a` 的 **title 属性**；
- 价格：`p.price_color` 的**纯文字**；
- 星级：`p.star-rating` 的 class 是 `['star-rating', 'Three']` 这种列表，**第二个词**才是星级。

In [6]:
for pod in soup.find_all('article', class_='product_pod'):
    title = pod.find('h3').find('a').get('title')                 # 书名在 a 的 title 属性
    price = pod.find('p', class_='price_color').get_text()        # 价格是纯文字
    star_class = pod.find('p', class_='star-rating').get('class')# class 列表
    print(title, '|', price, '| 星级 =', star_class[1])

A Light in the Attic | ¥51.77 | 星级 = Three
Tipping the Velvet | ¥53.74 | 星级 = One
Soumission | ¥50.10 | 星级 = Four
Sharp Objects | ¥47.82 | 星级 = Four


## 7.6 随堂小练习（自己动手，空格里写）

In [7]:
# TODO：找到底部 nav.pagination 里那个 <a>，打印它的 href（应是 books_page2.html）

In [8]:
# TODO：打印第一本书 availability 那段文字（提示：p 同时有 instock 和 availability 两个 class）

## 来源与延伸

- BeautifulSoup 官方文档（含 find / find_all 全部用法）：https://www.crummy.com/software/BeautifulSoup/bs4/doc/
- 下一课：CSS 选择器——用更短的一句话表达「我要找哪些节点」。